Git clone

In [ ]:
%cd /content
!git clone https://github.com/FaiazRahman/Multimodal-Fake-News-Detection.git
%cd Multimodal-Fake-News-Detection
!pip install -q sentence-transformers transformers pytorch-lightning timm pyyaml pillow

/content
fatal: destination path 'Multimodal-Fake-News-Detection' already exists and is not an empty directory.
/content/Multimodal-Fake-News-Detection


Replaces yaml.load() with yaml.safe_load() in the project files to safely read

In [ ]:
from pathlib import Path

for file in ["data_preprocessing.py", "run_training.py", "run_evaluation.py"]:
    p = Path(file)
    txt = p.read_text()
    txt = txt.replace("config = yaml.load(yaml_file)", "config = yaml.safe_load(yaml_file)")
    p.write_text(txt)

print("PyYAML patched.")

PyYAML patched.


Patched dataloader to handle truncated images safely

In [ ]:
from pathlib import Path

file = Path("dataloader.py")
text = file.read_text()

text = text.replace(
    "from PIL import Image",
    """from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES=True"""
)

text = text.replace(
"""
        df = None
        if not from_preprocessed_dataframe:
""",
"""
        self.dataset_type = dataset_type
        self.dir_to_save_dataframe = dir_to_save_dataframe

        self.saved_dataframe_filename_prefix=""

        if Modality(modality)==Modality.TEXT:
            self.saved_dataframe_filename_prefix="text"
        elif Modality(modality)==Modality.IMAGE:
            self.saved_dataframe_filename_prefix="image"
        elif Modality(modality)==Modality.TEXT_IMAGE:
            self.saved_dataframe_filename_prefix="text_image"
        elif Modality(modality)==Modality.TEXT_IMAGE_DIALOGUE:
            self.saved_dataframe_filename_prefix="text_image_dialogue"

        df=None

        if not from_preprocessed_dataframe:
"""
)

file.write_text(text)
print("dataloader patched.")

dataloader patched.


Updated trainer to use one GPU.

In [ ]:
from pathlib import Path

file = Path("run_training.py")
txt = file.read_text()
txt = txt.replace(
"""trainer = pl.Trainer(
            gpus=args.gpus,
            strategy="dp",
            callbacks=callbacks,
        )""",
"""trainer = pl.Trainer(
            accelerator="gpu",
            devices=1,
            callbacks=callbacks,
        )"""
)
file.write_text(txt)
print("Training trainer patched.")

Training trainer patched.


Updated evaluation to use one GPU

In [ ]:
from pathlib import Path

file = Path("run_evaluation.py")
txt = file.read_text()
txt = txt.replace(
"""trainer = pl.Trainer(
            gpus=args.gpus,
            strategy="dp",
            callbacks=callbacks,
        )""",
"""trainer = pl.Trainer(
            accelerator="gpu",
            devices=1,
            callbacks=callbacks,
        )"""
)
file.write_text(txt)
print("Evaluation trainer patched.")

Evaluation trainer patched.


Uploaded files from local computer to Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


Extracted dataset and prepared training data.

In [ ]:
!unzip -q dataset.zip
!mkdir -p data
!cp multimodal_only_samples/multimodal_train.tsv data/train_small.tsv
!ls data

image_downloader.py  requirements.txt  train_small.tsv


Created a smaller training dataset with 1500 samples.

In [ ]:
import pandas as pd

df = pd.read_csv("data/train_small.tsv", sep="\t")
small = df.head(1500)
small.to_csv("data/train_1500.tsv", sep="\t", index=False)

del df, small  
print("train_1500.tsv created")

train_1500.tsv created


Downloaded images for the 1500-sample dataset

In [ ]:
%cd data
!python image_downloader.py train_1500.tsv
%cd ..

/content/Multimodal-Fake-News-Detection/data
100% 1496/1500 [01:10<00:00, 23.16it/s]done
num_failed: 514
100% 1497/1500 [01:10<00:00, 21.15it/s]
/content/Multimodal-Fake-News-Detection


Check how many immage are downloaded

In [ ]:
!find data/images -type f | wc -l

983


Split dataset into training and testing sets.

In [ ]:
import pandas as pd

df = pd.read_csv("data/train_1500.tsv", sep="\t")
quick_train = df.iloc[:1200]
quick_test = df.iloc[1200:1500]

quick_train.to_csv("data/quick_train.tsv", sep="\t", index=False)
quick_test.to_csv("data/quick_test.tsv", sep="\t", index=False)

del df, quick_train, quick_test
print("split done")

split done


Created configuration file for model training.

In [ ]:
import yaml, os

os.makedirs("configs", exist_ok=True)

config = {
    "modality": "text-image",
    "num_classes": 2,
    "batch_size": 4,
    "learning_rate": 1e-4,
    "num_epochs": 1,
    "dropout_p": 0.1,
    "text_embedder": "all-distilroberta-v1",
    "dialogue_summarization_model": None,
    "train_data_path": "./data/quick_train.tsv",
    "test_data_path": "./data/quick_test.tsv",
    "gpus": [0],
    "trained_model_version": None,
    "trained_model_path": None,
}

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("quick_config.yaml ready")

quick_config.yaml ready


Preprocessed training and testing datasets successfully.

In [ ]:
!python data_preprocessing.py --train --test --config configs/quick_config.yaml

/content/Multimodal-Fake-News-Detection/data_preprocessing.py:37: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if args.config is not "":
Traceback (most recent call last):
  File "/content/Multimodal-Fake-News-Detection/data_preprocessing.py", line 15, in <module>
    from dataloader import MultimodalDataset, Modality
  File "/content/Multimodal-Fake-News-Detection/dataloader.py", line 16, in <module>
    ImageFile.LOAD_TRUNCATED_IMAGES=TrueFile
                                    ^^^^^^^^
NameError: name 'TrueFile' is not defined


In [ ]:
!git checkout -- dataloader.py
print("dataloader.py reset to original")

dataloader.py reset to original


Patched dataloader for images and datasets.

In [ ]:
from pathlib import Path

file = Path("dataloader.py")
text = file.read_text()

text = text.replace(
    "from PIL import Image",
    """from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES=True"""
)

text = text.replace(
"""
        df = None
        if not from_preprocessed_dataframe:
""",
"""
        self.dataset_type = dataset_type
        self.dir_to_save_dataframe = dir_to_save_dataframe

        self.saved_dataframe_filename_prefix=""

        if Modality(modality)==Modality.TEXT:
            self.saved_dataframe_filename_prefix="text"
        elif Modality(modality)==Modality.IMAGE:
            self.saved_dataframe_filename_prefix="image"
        elif Modality(modality)==Modality.TEXT_IMAGE:
            self.saved_dataframe_filename_prefix="text_image"
        elif Modality(modality)==Modality.TEXT_IMAGE_DIALOGUE:
            self.saved_dataframe_filename_prefix="text_image_dialogue"

        df=None

        if not from_preprocessed_dataframe:
"""
)

file.write_text(text)
print("dataloader patched.")

dataloader patched.


In [ ]:
!grep -n "LOAD_TRUNCATED_IMAGES" dataloader.py

13:ImageFile.LOAD_TRUNCATED_IMAGES=True


Preprocessed datasets after applying dataloader patch.

In [ ]:
!python data_preprocessing.py --train --test --config configs/quick_config.yaml

/content/Multimodal-Fake-News-Detection/data_preprocessing.py:37: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if args.config is not "":
INFO:datasets:TensorFlow version 2.20.0 available.
INFO:datasets:JAX version 0.7.2 available.
CUDA available: True
INFO:root:Namespace(train=True, test=True, from_dialogue_dataframe=None, dir_to_save_dataframe='data', config='configs/quick_config.yaml', train_data_path='./data/quick_train.tsv', test_data_path='./data/quick_test.tsv', modality='text-image', num_classes=2, text_embedder='all-distilroberta-v1', dialogue_summarization_model=None)
INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-distilroberta-v1/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/s

Reduced unnecessary logging messages during execution.

In [ ]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("root").setLevel(logging.WARNING)

Cleared memory and started model training.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

!python run_training.py --config configs/quick_config.yaml

Streaming output truncated to the last 5000 lines.
                                                               train_loss_epoch:
Batches: 100% 1/1 [00:00<00:00, 103.67it/s]
Epoch 24/999 ━━━━━━━━━━━━━╸━━ 165/195 0:00:35 •       4.69it/s v_num: 0.000     
                                      0:00:07                  train_loss_step: 
                                                               0.000            
                                                               train_loss_epoch:

Epoch 24/999 ━━━━━━━━━━━━━╸━━ 165/195 0:00:35 •       4.69it/s v_num: 0.000     
                                      0:00:07                  train_loss_step: 
                                                               0.000            
                                                               train_loss_epoch:
Batches:   0% 0/1 [00:00<?, ?it/s]
Epoch 24/999 ━━━━━━━━━━━━━╸━━ 165/195 0:00:35 •       4.69it/s v_num: 0.000     
                                      0:00:07              

In [2]:
!pwd
!ls

/content
sample_data


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/mmfnd_project', exist_ok=True)

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/mmfnd_project
!git clone https://github.com/FaiazRahman/Multimodal-Fake-News-Detection.git
%cd Multimodal-Fake-News-Detection

/content/drive/MyDrive/mmfnd_project
Cloning into 'Multimodal-Fake-News-Detection'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 259 (delta 26), reused 57 (delta 26), pack-reused 202 (from 1)
Receiving objects: 100% (259/259), 268.37 KiB | 5.96 MiB/s, done.
Resolving deltas: 100% (148/148), done.
/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection


In [5]:
!pip install -q sentence-transformers transformers pytorch-lightning timm pyyaml pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.5 MB/s eta 0:00:00


In [7]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("root").setLevel(logging.WARNING)

In [8]:
from pathlib import Path

for file in ["data_preprocessing.py", "run_training.py", "run_evaluation.py"]:
    p = Path(file)
    txt = p.read_text()
    txt = txt.replace("config = yaml.load(yaml_file)", "config = yaml.safe_load(yaml_file)")
    p.write_text(txt)

print("PyYAML patched.")

PyYAML patched.


In [9]:
from pathlib import Path

file = Path("dataloader.py")
text = file.read_text()

text = text.replace(
    "from PIL import Image",
    """from PIL import Image
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES=True"""
)

text = text.replace(
"""
        df = None
        if not from_preprocessed_dataframe:
""",
"""
        self.dataset_type = dataset_type
        self.dir_to_save_dataframe = dir_to_save_dataframe

        self.saved_dataframe_filename_prefix=""

        if Modality(modality)==Modality.TEXT:
            self.saved_dataframe_filename_prefix="text"
        elif Modality(modality)==Modality.IMAGE:
            self.saved_dataframe_filename_prefix="image"
        elif Modality(modality)==Modality.TEXT_IMAGE:
            self.saved_dataframe_filename_prefix="text_image"
        elif Modality(modality)==Modality.TEXT_IMAGE_DIALOGUE:
            self.saved_dataframe_filename_prefix="text_image_dialogue"

        df=None

        if not from_preprocessed_dataframe:
"""
)

file.write_text(text)
print("dataloader patched.")

dataloader patched.


In [10]:
from pathlib import Path

file = Path("run_training.py")
txt = file.read_text()
txt = txt.replace(
"""trainer = pl.Trainer(
            gpus=args.gpus,
            strategy="dp",
            callbacks=callbacks,
        )""",
"""trainer = pl.Trainer(
            accelerator="gpu",
            devices=1,
            max_epochs=args.num_epochs,
            callbacks=callbacks,
        )"""
)
file.write_text(txt)
print("Training trainer patched (with max_epochs fix).")

Training trainer patched (with max_epochs fix).


In [11]:
from pathlib import Path

file = Path("run_evaluation.py")
txt = file.read_text()
txt = txt.replace(
"""trainer = pl.Trainer(
            gpus=args.gpus,
            strategy="dp",
            callbacks=callbacks,
        )""",
"""trainer = pl.Trainer(
            accelerator="gpu",
            devices=1,
            callbacks=callbacks,
        )"""
)
file.write_text(txt)
print("Evaluation trainer patched.")

Evaluation trainer patched.


In [12]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [13]:
!unzip -q dataset.zip
!mkdir -p data
!cp multimodal_only_samples/multimodal_train.tsv data/train_small.tsv
!ls data

image_downloader.py  requirements.txt  train_small.tsv


In [14]:
import pandas as pd

df = pd.read_csv("data/train_small.tsv", sep="\t")
small = df.head(1500)
small.to_csv("data/train_1500.tsv", sep="\t", index=False)

del df, small
print("train_1500.tsv created")

train_1500.tsv created


In [15]:
%cd data
!python image_downloader.py train_1500.tsv
%cd ..

/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/data
 99% 1491/1500 [00:38<00:00, 35.49it/s]done
num_failed: 514
100% 1497/1500 [00:38<00:00, 38.74it/s]
/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection


In [16]:
!find data/images -type f | wc -l

983


In [17]:
import pandas as pd

df = pd.read_csv("data/train_1500.tsv", sep="\t")
quick_train = df.iloc[:1200]
quick_test = df.iloc[1200:1500]

quick_train.to_csv("data/quick_train.tsv", sep="\t", index=False)
quick_test.to_csv("data/quick_test.tsv", sep="\t", index=False)

del df, quick_train, quick_test
print("split done")

split done


In [18]:
import yaml, os

os.makedirs("configs", exist_ok=True)

config = {
    "modality": "text-image",
    "num_classes": 2,
    "batch_size": 4,
    "learning_rate": 1e-4,
    "num_epochs": 1,
    "dropout_p": 0.1,
    "text_embedder": "all-distilroberta-v1",
    "dialogue_summarization_model": None,
    "train_data_path": "./data/quick_train.tsv",
    "test_data_path": "./data/quick_test.tsv",
    "gpus": [0],
    "trained_model_version": None,
    "trained_model_path": None,
}

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("quick_config.yaml ready")

quick_config.yaml ready


In [19]:
!python data_preprocessing.py --train --test --config configs/quick_config.yaml > preprocess_log.txt 2>&1
!tail -15 preprocess_log.txt

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distilroberta-v1/842eaed40bee4d61673a81c92d5689a8fed7a09f/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distilroberta-v1/842eaed40bee4d61673a81c92d5689a8fed7a09f/special_tokens_map.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-distilroberta-v1/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-distilroberta-v1/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distilroberta-v1/842eaed40bee4d61673a81c92d5689a8fed7a09f/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/

In [20]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

!python run_training.py --config configs/quick_config.yaml > train_log.txt 2>&1
!tail -30 train_log.txt

^C
Batches: 100%|██████████| 1/1 [00:00<00:00, 10.97it/s]
0.4133458137512207
Batches: 100%|██████████| 1/1 [00:00<00:00, 22.02it/s]
0.42998141050338745
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.71it/s]
0.42743468284606934
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.70it/s]
0.4850628972053528
Batches: 100%|██████████| 1/1 [00:00<00:00, 24.95it/s]

Detected KeyboardInterrupt, attempting graceful shutdown ...
Epoch 1/999 ━━━━━━           74/195 0:08:17 • 0:13:56 0.14it/s v_num: 0.000     
                                                               train_loss_step: 
                                                               0.485            
                                                               train_loss_epoch:
                                                               0.687            


In [21]:
!tail -50 train_log.txt

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.12it/s]
0.44029098749160767
Batches: 100%|██████████| 1/1 [00:00<00:00, 24.03it/s]
0.7080197930335999
Batches: 100%|██████████| 1/1 [00:00<00:00, 16.85it/s]
0.39497798681259155
Batches: 100%|██████████| 1/1 [00:00<00:00, 19.34it/s]
1.1741851568222046
Batches: 100%|██████████| 1/1 [00:00<00:00, 10.97it/s]
0.4133458137512207
Batches: 100%|██████████| 1/1 [00:00<00:00, 22.02it/s]
0.42998141050338745
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.71it/s]
0.42743468284606934
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.70it/s]
0.4850628972053528
Batches: 100%|██████████| 1/1 [00:00<00:00, 24.95it/s]

Detected KeyboardInterrupt, attempting graceful shutdown ...
Epoch 1/999 ━━━━━━           74/195 0:08:17 • 0:13:56 0.14it/s v_num: 0.000     
                                                               train_loss_step: 
                                                               0.485            
                                            

In [22]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

!nvidia-smi

CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found


In [23]:
import pandas as pd

df = pd.read_csv("data/train_1500.tsv", sep="\t")
quick_train = df.iloc[:150]
quick_test = df.iloc[150:200]

quick_train.to_csv("data/quick_train.tsv", sep="\t", index=False)
quick_test.to_csv("data/quick_test.tsv", sep="\t", index=False)

del df, quick_train, quick_test
print("small split done")

small split done


In [24]:
import yaml

with open("configs/quick_config.yaml") as f:
    config = yaml.safe_load(f)

config["batch_size"] = 8
config["train_data_path"] = "./data/quick_train.tsv"
config["test_data_path"] = "./data/quick_test.tsv"

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("config updated for CPU run")

config updated for CPU run


In [25]:
!python data_preprocessing.py --train --test --config configs/quick_config.yaml > preprocess_log.txt 2>&1
!tail -15 preprocess_log.txt

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-distilroberta-v1/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distilroberta-v1/842eaed40bee4d61673a81c92d5689a8fed7a09f/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-distilroberta-v1/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-distilroberta-v1/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-distilroberta-v1/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distil

In [27]:
!du -sh /content/drive/MyDrive/mmfnd_project/* 2>/dev/null
!du -sh /content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/lightning_logs 2>/dev/null

958M	/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection
683M	/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/lightning_logs


In [28]:
!rm -rf /content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/multimodal_only_samples
!rm -f /content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/dataset.zip
!rm -f /content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/data/train_small.tsv
print("cleaned up extra files")

cleaned up extra files


In [29]:
from pathlib import Path

file = Path("run_training.py")
txt = file.read_text()

# পুরনো যেকোনো Trainer definition রিমুভ করে নতুন করে বসাই
import re
txt = re.sub(
    r'trainer = pl\.Trainer\([^)]*\)',
    '''trainer = pl.Trainer(
            accelerator="auto",
            devices=1,
            max_epochs=1,
            enable_checkpointing=True,
            callbacks=callbacks,
        )''',
    txt
)

file.write_text(txt)
print("Trainer forcibly patched with max_epochs=1")

!grep -n "max_epochs" run_training.py

Trainer forcibly patched with max_epochs=1
139:            max_epochs=1,
147:            max_epochs=1,


In [30]:
!du -sh /content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection
!ls data/quick_train.tsv data/quick_test.tsv

706M	/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection
data/quick_test.tsv  data/quick_train.tsv


In [31]:
!python run_training.py --config configs/quick_config.yaml > train_log.txt 2>&1
!tail -30 train_log.txt

Do it fast
Batches: 100%|██████████| 1/1 [00:00<00:00, 22.56it/s]
0.6861352920532227
Batches: 100%|██████████| 1/1 [00:00<00:00, 18.62it/s]
`Trainer.fit` stopped: `max_epochs=1` reached.
0.8160454630851746
Training done...
0.703934371471405
0.6915194392204285
0.6898244619369507
0.6928434371948242
0.6942573189735413
0.6830474734306335
0.698245644569397
0.6607537269592285
0.6465268731117249
0.6712751388549805
0.7459408640861511
0.6861352920532227
0.8160454630851746
Epoch 0/0  ━━━━━━━━━━━━━━━━━ 13/13 0:02:39 • 0:00:00 0.10it/s v_num: 2.000      
                                                              train_loss_step:  
                                                              0.816             
                                                              train_loss_epoch: 
                                                              0.691             


In [32]:
!ls lightning_logs

log.log  version_0  version_1  version_2


In [34]:
import yaml

with open("configs/quick_config.yaml") as f:
    config = yaml.safe_load(f)

config["trained_model_version"] = 2

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("config updated with version:", config["trained_model_version"])

config updated with version: 2


In [35]:
!python run_evaluation.py --config configs/quick_config.yaml > eval_log.txt 2>&1
!tail -30 eval_log.txt

INFO:root:./lightning_logs/version_2/checkpoints/epoch=0-step=13.ckpt
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [36]:
from pathlib import Path

file = Path("model.py")
txt = file.read_text()

# Fix 1: remove .cuda() calls (breaks on CPU-only runtime)
txt = txt.replace(
    "'test_acc': torch.tensor(accuracy).cuda()",
    "'test_acc': torch.tensor(accuracy)"
)

# Fix 2: accumulate outputs manually in test_step
txt = txt.replace(
    """        print(loss.item(), output['test_acc'])
        return output""",
    """        print(loss.item(), output['test_acc'])
        if not hasattr(self, "test_step_outputs"):
            self.test_step_outputs = []
        self.test_step_outputs.append(output)
        return output"""
)

# Fix 3: replace old test_epoch_end(self, outputs) with new on_test_epoch_end(self)
txt = txt.replace(
    """    # Optional for pl.LightningModule
    def test_epoch_end(self, outputs):
        avg_loss = torch.stack([x["test_loss"] for x in outputs]).mean()
        avg_accuracy = torch.stack([x["test_acc"] for x in outputs]).mean()
        logs = {
            'test_loss': avg_loss,
            'test_acc': avg_accuracy
        }

        # pl.LightningModule has some issues displaying the results automatically
        # As a workaround, we can store the result logs as an attribute of the
        # class instance and display them manually at the end of testing
        # https://github.com/PyTorchLightning/pytorch-lightning/issues/1088
        self.test_results = logs

        return {
            'avg_test_loss': avg_loss,
            'avg_test_acc': avg_accuracy,
            'log': logs,
            'progress_bar': logs
        }""",
    """    # Optional for pl.LightningModule
    def on_test_epoch_end(self):
        outputs = self.test_step_outputs
        avg_loss = torch.stack([x["test_loss"] for x in outputs]).mean()
        avg_accuracy = torch.stack([x["test_acc"] for x in outputs]).mean()
        logs = {
            'test_loss': avg_loss,
            'test_acc': avg_accuracy
        }
        self.test_results = logs
        self.test_step_outputs = []"""
)

file.write_text(txt)
print("model.py patched.")

# verify
!grep -n "def test_epoch_end\|def on_test_epoch_end\|.cuda()" model.py

model.py patched.
154:    def on_test_epoch_end(self):
310:    def on_test_epoch_end(self):


In [37]:
!python run_evaluation.py --config configs/quick_config.yaml > eval_log.txt 2>&1
!tail -30 eval_log.txt

Batches: 100%|██████████| 1/1 [00:00<00:00, 17.85it/s]
0.6956006288528442 tensor(0.5000)
Batches: 100%|██████████| 1/1 [00:00<00:00, 18.09it/s]
0.6199224591255188 tensor(0.7500)
Batches: 100%|██████████| 1/1 [00:00<00:00, 10.47it/s]
0.7064570784568787 tensor(0.5000)
Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 0:00:16 • 0:00:00 0.23it/s 
INFO:root:./data/quick_test.tsv
INFO:root:./lightning_logs/version_2/checkpoints/epoch=0-step=13.ckpt
INFO:root:{'test_loss': tensor(0.6694), 'test_acc': tensor(0.5625)}
./data/quick_test.tsv
./lightning_logs/version_2/checkpoints/epoch=0-step=13.ckpt
{'test_loss': tensor(0.6694), 'test_acc': tensor(0.5625)}


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection
!ls data

/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection
image_downloader.py  requirements.txt
images		     test__text_image__dataframe.pkl
quick_test.tsv	     train_1500.tsv
quick_train.tsv      train__text_image__dataframe.pkl


In [5]:
import pandas as pd

df = pd.read_csv("data/train_1500.tsv", sep="\t")
quick_train = df.iloc[:1200]
quick_test = df.iloc[1200:1500]

quick_train.to_csv("data/quick_train.tsv", sep="\t", index=False)
quick_test.to_csv("data/quick_test.tsv", sep="\t", index=False)

del df, quick_train, quick_test
print("bigger split done")

bigger split done


In [6]:
import yaml

with open("configs/quick_config.yaml") as f:
    config = yaml.safe_load(f)

config["num_epochs"] = 3
config["batch_size"] = 8
config["train_data_path"] = "./data/quick_train.tsv"
config["test_data_path"] = "./data/quick_test.tsv"

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("config updated:", config["num_epochs"], "epochs")

config updated: 3 epochs


In [7]:
from pathlib import Path

file = Path("run_training.py")
txt = file.read_text()
txt = txt.replace("max_epochs=1,", "max_epochs=args.num_epochs,")
file.write_text(txt)

!grep -n "max_epochs" run_training.py

139:            max_epochs=args.num_epochs,
147:            max_epochs=args.num_epochs,


In [8]:
!pip install -q sentence-transformers transformers pytorch-lightning timm pyyaml pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.4 MB/s eta 0:00:00


In [9]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("root").setLevel(logging.WARNING)

In [10]:
!python data_preprocessing.py --train --test --config configs/quick_config.yaml > preprocess_log.txt 2>&1
!tail -10 preprocess_log.txt

INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distilroberta-v1/842eaed40bee4d61673a81c92d5689a8fed7a09f/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-distilroberta-v1 "HTTP/1.1 200 OK"
INFO:root:Preprocessed dataframe saved to data/train__text_image__dataframe.pkl
INFO:root:Train dataset size: 778
INFO:root:<dataloader.MultimodalDataset object at 0x7abbde27b920>
INFO:root:Preprocessed dataframe saved to data/test__text_image__dataframe.pkl
INFO:root:Test dataset size: 205
INFO:root:<dataloader.MultimodalDataset object at 0x7abbc010bf20>
Preprocessed dataframe saved to data/train__text_image__dataframe.pkl
Preprocessed dataframe saved to data/test__text_image__dataframe.pkl


In [11]:
!python run_training.py --config configs/quick_config.yaml > train_log.txt 2>&1
!tail -30 train_log.txt

0.6379111409187317
0.33882805705070496
0.19695279002189636
0.4151551127433777
0.15011224150657654
0.7817937135696411
0.3442191183567047
0.3721330463886261
0.6215277314186096
0.4260820150375366
1.3753807544708252
0.34319013357162476
0.5679706335067749
0.26623284816741943
0.2770683765411377
0.4698256254196167
0.3859613239765167
0.30855536460876465
0.5177009105682373
0.7596242427825928
0.35658931732177734
0.37468788027763367
0.35303395986557007
0.32234758138656616
0.9502857327461243
Epoch 2/2  ━━━━━━━━━━━━━━━━━ 98/98 0:00:35 • 0:00:00 2.69it/s v_num: 4.000      
                                                              train_loss_step:  
                                                              0.950             
                                                              train_loss_epoch: 
                                                              0.413             


In [12]:
!ls lightning_logs

log.log  version_0  version_1  version_2  version_3  version_4


In [13]:
import yaml

with open("configs/quick_config.yaml") as f:
    config = yaml.safe_load(f)

config["trained_model_version"] = 4

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("version set to:", config["trained_model_version"])

version set to: 4


In [14]:
!python run_evaluation.py --config configs/quick_config.yaml > eval_log.txt 2>&1
!tail -10 eval_log.txt

Batches: 100%|██████████| 1/1 [00:00<00:00, 96.57it/s]
0.3391626477241516 tensor(0.8000)
Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26/26 0:00:06 • 0:00:00 4.17it/s 
INFO:root:./data/quick_test.tsv
INFO:root:./lightning_logs/version_4/checkpoints/epoch=2-step=294.ckpt
INFO:root:{'test_loss': tensor(0.7820, device='cuda:0'), 'test_acc': tensor(0.6125)}
./data/quick_test.tsv
./lightning_logs/version_4/checkpoints/epoch=2-step=294.ckpt
{'test_loss': tensor(0.7820, device='cuda:0'), 'test_acc': tensor(0.6125)}


In [15]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [17]:
from google.colab import files
uploaded = files.upload()

Saving dataset.zip to dataset.zip


In [18]:
!unzip -q dataset.zip
import pandas as pd

df = pd.read_csv("multimodal_only_samples/multimodal_train.tsv", sep="\t")
big_sample = df.head(5000)
big_sample.to_csv("data/train_5000.tsv", sep="\t", index=False)

del df, big_sample
print("train_5000.tsv created")

train_5000.tsv created


In [19]:
%cd data
!python image_downloader.py train_5000.tsv
%cd ..
!find data/images -type f | wc -l

/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection/data
100% 4988/5000 [04:27<00:01, 11.92it/s]done
num_failed: 1679
100% 4988/5000 [04:27<00:00, 18.62it/s]
/content/drive/MyDrive/mmfnd_project/Multimodal-Fake-News-Detection
3309


In [20]:
import pandas as pd

df = pd.read_csv("data/train_5000.tsv", sep="\t")
quick_train = df.iloc[:4000]
quick_test = df.iloc[4000:5000]

quick_train.to_csv("data/quick_train.tsv", sep="\t", index=False)
quick_test.to_csv("data/quick_test.tsv", sep="\t", index=False)

del df, quick_train, quick_test
print("bigger split done")

bigger split done


In [21]:
import yaml

with open("configs/quick_config.yaml") as f:
    config = yaml.safe_load(f)

config["num_epochs"] = 5
config["batch_size"] = 16
config["train_data_path"] = "./data/quick_train.tsv"
config["test_data_path"] = "./data/quick_test.tsv"

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("config updated:", config)

config updated: {'batch_size': 16, 'dialogue_summarization_model': None, 'dropout_p': 0.1, 'gpus': [0], 'learning_rate': 0.0001, 'modality': 'text-image', 'num_classes': 2, 'num_epochs': 5, 'test_data_path': './data/quick_test.tsv', 'text_embedder': 'all-distilroberta-v1', 'train_data_path': './data/quick_train.tsv', 'trained_model_path': None, 'trained_model_version': 4}


In [22]:
!python data_preprocessing.py --train --test --config configs/quick_config.yaml > preprocess_log.txt 2>&1
!tail -10 preprocess_log.txt

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-distilroberta-v1/842eaed40bee4d61673a81c92d5689a8fed7a09f/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-distilroberta-v1 "HTTP/1.1 200 OK"
INFO:root:Preprocessed dataframe saved to data/train__text_image__dataframe.pkl
INFO:root:Train dataset size: 2629
INFO:root:<dataloader.MultimodalDataset object at 0x7c6c345ec230>
INFO:root:Preprocessed dataframe saved to data/test__text_image__dataframe.pkl
INFO:root:Test dataset size: 680
INFO:root:<dataloader.MultimodalDataset object at 0x7c6c34425460>
Preprocessed dataframe saved to data/train__text_image__dataframe.pkl
Preprocessed dataframe saved to data/test__text_image__dataframe.pkl


In [24]:
!python run_training.py --config configs/quick_config.yaml > train_log.txt 2>&1
!tail -30 train_log.txt

0.08585752546787262
0.16181142628192902
0.23987822234630585
0.1954137682914734
1.0516796112060547
0.2760615646839142
0.38538748025894165
0.12282906472682953
0.17272605001926422
0.22632645070552826
0.13918337225914001
0.24382661283016205
0.27663347125053406
0.18114425241947174
0.11493094265460968
0.07809002697467804
0.6213382482528687
0.2082681655883789
0.6504316329956055
0.3551390469074249
0.10612653195858002
0.1471591740846634
0.09557612240314484
0.3994781970977783
0.07510586827993393
Epoch 4/4  ━━━━━━━━━━━━━━━━ 165/165 0:01:49 • 0:00:00 1.55it/s v_num: 7.000     
                                                               train_loss_step: 
                                                               0.075            
                                                               train_loss_epoch:
                                                               0.261            


In [25]:
!ls lightning_logs

log.log    version_1  version_3  version_5  version_7
version_0  version_2  version_4  version_6


In [26]:
import yaml

with open("configs/quick_config.yaml") as f:
    config = yaml.safe_load(f)

config["trained_model_version"] = 7

with open("configs/quick_config.yaml", "w") as f:
    yaml.dump(config, f)

print("version set to:", config["trained_model_version"])

version set to: 7


In [27]:
!python run_evaluation.py --config configs/quick_config.yaml > eval_log.txt 2>&1
!tail -10 eval_log.txt

Batches: 100%|██████████| 1/1 [00:00<00:00, 153.92it/s]
0.7202746868133545 tensor(0.7500)
Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43/43 0:00:21 • 0:00:00 1.98it/s 
INFO:root:./data/quick_test.tsv
INFO:root:./lightning_logs/version_7/checkpoints/epoch=4-step=825.ckpt
INFO:root:{'test_loss': tensor(0.8260, device='cuda:0'), 'test_acc': tensor(0.7020)}
./data/quick_test.tsv
./lightning_logs/version_7/checkpoints/epoch=4-step=825.ckpt
{'test_loss': tensor(0.8260, device='cuda:0'), 'test_acc': tensor(0.7020)}
